### Step 1: Import Libraraies & API Keys

In [28]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import json

load_dotenv()

OpenAI_API_KEY = os.getenv("OPENAI_API_KEY")

if  OpenAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY is missing")

### Step 2: Simple RAG with Guardrails and dynamic context Injection

In [29]:
system_message = """You are a digital twin of Tarun Maheswaram. When people talk to you, you respond AS Tarun Maheswaram — in first person, using his voice, personality, experience, and knowledge.

use only factual information giuven to you .

Here's information about Tarun Maheswaram to help you embody him:

Tarun Maheswaram is a Business Intelligence (BI) Developer with over 9 years of experience helping organizations transform raw data into meaningful business insights. He specializes in designing dashboards, building reporting solutions, optimizing databases, and enabling data-driven decision making.

He holds a Master's degree in Information Technology and has built his career around data analytics, business intelligence, and modern reporting platforms.

Throughout his career, he has worked with several organizations including:

 I have started pursuing my bachelor's degree in ECE in India in 2011.
• Blue Cross of Idaho, where he worked for over six years as a BI Developer building enterprise reporting solutions and supporting healthcare analytics.
• MWI Animal Health, developing reporting and analytics solutions that improved business visibility and operational reporting.
• Pennsylvania Transformer Technology, where he worked as a Power BI and Tableau Developer, creating interactive dashboards and business intelligence solutions.

His technical expertise includes:

- Microsoft SQL Server
- Advanced SQL
- Power BI
- Tableau
- SSRS (SQL Server Reporting Services)
- Data Warehousing
- ETL processes
- Data Modeling
- Dashboard Design
- Performance Optimization
- Business Intelligence Architecture

He enjoys solving complex business problems using data. He believes every dataset tells a story, and his goal is to uncover insights that help businesses make better decisions. He especially enjoys taking messy, disconnected data and transforming it into clear, interactive dashboards that executives and stakeholders can understand.

Recently, he has expanded his skills into Artificial Intelligence and Generative AI. He enjoys building AI-powered applications using Python, OpenAI APIs, Gradio, Retrieval-Augmented Generation (RAG), and Large Language Models. He likes combining his background in business intelligence with AI to create practical solutions that automate work and improve decision making.

What drives him:
He genuinely enjoys learning new technologies and continuously improving his skills. He's naturally curious and enjoys experimenting with new tools, especially in AI, automation, analytics, and cloud technologies. He believes technology should simplify people's work rather than make it more complicated.

His approach:
Practical, analytical, and solution-oriented. He prefers explaining technical concepts in simple language with real-world examples instead of unnecessary jargon. When solving problems, he thinks step by step and values clean, maintainable solutions over overly complex ones.

Communication style:
Friendly, approachable, patient, and professional. He enjoys mentoring others and sharing knowledge without sounding overly formal. He explains things clearly and adapts his explanations depending on the person's technical background.

When responding:
- Always answer in first person.
- Respond naturally as if you are Tarun Maheswaram, not an AI assistant.
- Draw from your professional experience whenever appropriate.
- If someone asks for career advice, data analytics, Power BI, Tableau, SQL, SSRS, or AI, answer from your own experience.
- If you don't know something, be honest instead of inventing information.
- If you asked about something not mentioned in the context, respond with "I don't want to talk about it.
- Never provide a wrong answer which is not in the context and topic context. If you don't know something, be honest instead of inventing information.
- Never assume or make up information about Tarun Maheswaram. If you don't know something, respond with "I don't want to talk about it.
- Keep responses conversational, helpful, and authentic."""

In [30]:
Topic_Context = {

    "education": {
        "keywords": ["2011", "college", "bachelor", "degree", "ece", "engineering", "study"],
        "context": "***I have started pursuing my bachelor's degree in ECE in India in 2011.***"
    },

    "cooking": {
        "keywords": ["cook", "cooking", "recipe", "recipes", "food", "cuisine"],
        "context": "***I love cooking and experimenting with new recipes. I enjoy trying out different cuisines and flavors, and I find it to be a great way to relax and express creativity. Cooking is not just about preparing food for me; it's also about the joy of sharing meals with family and friends, and creating memorable experiences around the dining table.***"
    },

    "cars": {
        "keywords": ["car", "cars", "muscle", "vehicle", "automobile", "mustang"],
        "context": "***I love muscle cars and have a deep appreciation for their design, engineering, and performance. I enjoy learning about different car models, their specifications, and the history behind iconic vehicles. Attending car shows and events is something I look forward to, as it allows me to connect with fellow enthusiasts and stay updated on the latest trends in the automotive world.***"
    },

    "fitness": {
        "keywords": ["fitness", "exercise", "workout", "gym", "health", "nutrition"],
        "context": "***I believe maintaining physical fitness is important for both health and productivity. I enjoy exercising regularly and learning about nutrition and healthy habits.***"
    },

    "travel": {
        "keywords": ["travel", "trip", "vacation", "culture", "places", "destination"],
        "context": "***I enjoy traveling and experiencing different cultures, local cuisines, and historical places. Exploring new destinations gives me fresh perspectives and memorable experiences.***"
    },

    "music": {
        "keywords": ["music", "song", "songs", "artist", "genre"],
        "context": "***I enjoy listening to music while working or relaxing. My taste varies depending on my mood, and I appreciate discovering new artists and genres.***"
    },

    "gaming": {
        "keywords": ["game", "games", "gaming", "video game", "play"],
        "context": "***I enjoy playing video games as a way to unwind and challenge myself. I appreciate games that offer immersive storytelling, strategic thinking, and engaging gameplay mechanics. Gaming also allows me to connect with friends and the gaming community, sharing experiences and tips.***"
    }
}

### Step 3: Prepare the list of tools for the LLM

In [42]:
tools = []

### Step 3a: Add tool calling functionality (Pushover)

In [43]:
pushover_user =  os.getenv("PUSHOVER_USER")
pushover_token =  os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


#Create send notification function
import requests

def send_notification(message: str):
    payload = {"user": pushover_user,"token": pushover_token,"message": message}
    requests.post(pushover_url, data=payload)


#Test the notification function
#send_notification("Hello Tarun,Have a good day!")



#Describe Pushover as an LLM Tool
send_notification_funtion = {
    "name":"send_notification",
    "description":"sends a push notification to the real-world version of you via pushover on mobile. Use this to alert the user about important events, completed tasks or time-sensitive information. ",
    "parameters": {
        "type":"object",
        "properties": {
            "message":{
                "type":"string",
                "description": "The notification message to send to user's device"
            }
        },
        "required":["message"]
    }
}


# Add Pushover to the list of tools for the LLM
tools.append({"type":"function","function":send_notification_funtion})

### Step 3b: Add dice rolling functionality

In [44]:
import random 

#Simulate rolling a single six-sided dice
def dice_roll():
    result= random.randint(1,6)
    return result

#Describe function for LLM
roll_dice_function = {
    "name":"dice_roll",
    "description":"Simulates rolling a dice and returns the result. Use this when user wants to roll the dice. ",
    "parameters": {
        "type":"object",
        "properties": {},
        "required":[]
            
            }
        }

#Add funtion to list of tools of LLM
tools.append({"type":"function","function":send_notification_funtion})

### Step 4: Fucntion top handle LLM Tool Calls

In [36]:
def handle_tool_call(tool_calls):
    tool_results=[]


    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f"calling function {function_name}") #for future debugging
        
        #route to appropriate function based on fucntion name
        if function_name == "send_notification":
            #Actaully send the notification i.e call the tool
            send_notification(args["message"])
            content =f"Notification Sent: {args['message']}"
            #print(f"send_notification:{args["message"]}")
        elif function_name =="dice_roll":
             content = f"Rolled: {dice_roll()}"
                #elif funtion_name =="insert_function_name_3"
        #   content =insert_function_name_3(args["message]"])
                #elif funtion_name =="insert_function_name_4"
        #   content =insert_function_name_4(args["message]"])
        else:
            content = f"Unknown funtion: {function_name}"

        tool_call_result={
        "role":"tool",
        "content":f"Notification sent: {args['message']}",
        "tool_call_id": tool_call.id

    }
    
        tool_results.append(tool_call_result)

    return tool_results


### Step 5: Fucntion to Process the conversation Turn

In [ ]:
def respond_ai(message, history):

    #Dynamic Injection Code based on the keywords in the message
    system_message_enhanced = system_message
    user_message = message.lower()

    for topic, data in Topic_Context.items():
        for keyword in data["keywords"]:
         if keyword in user_message:
            system_message_enhanced += "\n\n" + data["context"]
            break
       





    #AS Usual code
    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OpenAI_API_KEY)
    response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
    )
    message = response.choices[0].message
  
    #Check if model wants to call a tool 
    
    while message.tool_calls:
        from pprint import pprint
        pprint(message.tool_calls)
        tool_result = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
        messages.append(message)
        messages.extend(tool_result) #changed from append() to extend()when we swtiched to multiple tool call handling.
        
        response =client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        message =response.choices[0].message

        #maybe consider adding protection from infinite consecutive tool calling
    return(message.content)
   
        
 



### Step 6: Launch Gradio

In [46]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True) 

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


[ChatCompletionMessageFunctionToolCall(id='call_zFDJ8xl3zwja9DTYOqgG1KiM', function=Function(arguments='{"message": "First roll: rolled 3 dice with values 4, 2, 6. Highest roll is 6."}', name='send_notification'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_Rj6lWvCf5lN3AYH8TZglgCDM', function=Function(arguments='{"message": "Second roll: rolled 3 dice with values 5, 3, 1. Highest roll of all 6 dice (4, 2, 6, 5, 3, 1) is 6."}', name='send_notification'), type='function')]
